In [1]:
import os
import re
from functools import reduce
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import torch
from bs4 import BeautifulSoup
from bs4.element import NavigableString
from huggingface_hub import InferenceClient
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    pipeline,
)

from pandas.api.types import (
    is_datetime64_any_dtype,
    is_numeric_dtype,
)
from pandas.core.dtypes.dtypes import DatetimeTZDtype

tqdm.pandas()

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Change working directory to project root
PROJECT_ROOT = Path().absolute().parent if Path().absolute().name == 'processors' else Path().absolute()
os.chdir(PROJECT_ROOT)
!pwd

/Users/phatvu/Documents/Dev-Drive-Local/crypto-price-forecaster-glm


In [3]:
# Import project configuration
import sys
sys.path.append('.')
from config import *

In [4]:
hf_key = os.getenv("hf_key")
if hf_key is None:
    print("Warning: hf_key not found in environment variables")
else:
    print("hf_key loaded from environment variables")

hf_key loaded from environment variables


## Utils

### Convert Timestamp

In [5]:
def convert_timestamp(
    df: pd.DataFrame,
    col: str,
    tz: str = "UTC",
    round_to: str = "s"
) -> pd.DataFrame:
    """Convert and normalize a timestamp column to UTC, with optional rounding.
       Includes preprocessing for mixed ISO 8601 strings (with or without microseconds).
    """

    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found.")

    df_copy = df.copy()
    s = df_copy[col]
    dtype = s.dtype

    # datetime-like, tz-aware or tz-naive
    if is_datetime64_any_dtype(s) or isinstance(dtype, DatetimeTZDtype):
        ts = pd.to_datetime(s, utc=True).dt.tz_convert(tz)

    # numeric epoch
    elif is_numeric_dtype(s):
        s_nonnull = s.dropna()
        if s_nonnull.empty:
            raise ValueError("Timestamp column is empty.")

        max_val = s_nonnull.max()
        # Determine unit based on magnitude
        if max_val > 1e17:
            unit = "ns"
        elif max_val > 1e14:
            unit = "us"
        elif max_val > 1e11:
            unit = "ms"
        else:
            unit = "s"

        ts = pd.to_datetime(s, unit=unit, utc=True).dt.tz_convert(tz)

    # string / mixed formats (including the problematic microsecond format)
    else:
        # Preprocess: normalize strings before parsing
        s_str = s.astype(str)
        
        # Remove microseconds to avoid parsing errors when Pandas infers format
        # Pattern: find .[digits]...[digits]+00:00 (e.g., .844001+00:00) and replace with +00:00
        s_clean = s_str.str.replace(r'\.\d{3,6}\+00:00', r'+00:00', regex=True)
        
        # Parse to datetime
        ts = pd.to_datetime(s_clean, utc=True, errors="coerce")
        
        if ts.isna().all():
            raise ValueError(f"Cannot parse timestamps in column '{col}'.")
        ts = ts.dt.tz_convert(tz)

    # rounding (use lowercase 's', 'ms', 'us', etc.)
    if round_to is not None:
        ts = ts.dt.round(round_to)

    df_copy[col] = ts
    return df_copy

### Crawl News

#### Extract html

In [6]:
def extract_article(source: str) -> dict:
    """
    Extracts structured content from Bitcoin Magazine HTML.
    Optimized for 2012-2025 layouts with proper paragraph separation.
    """
    # 1. Load content
    html_content = ""
    if os.path.exists(source) and os.path.isfile(source):
        try:
            with open(source, "r", encoding="utf-8") as f:
                html_content = f.read()
        except Exception:
            return {}
    else:
        html_content = source

    soup = BeautifulSoup(html_content, 'html.parser')
    data = {}

    # 2. Metadata extraction
    meta_title = soup.find("meta", property="og:title")
    title_content = meta_title.get("content", "") if meta_title else ""
    data['title'] = str(title_content).strip() if title_content else ""
    if not data['title']:
        t_tag = soup.find("title")
        data['title'] = t_tag.get_text(strip=True) if t_tag else "Unknown"
    meta_auth = soup.find("meta", attrs={"name": "author"})
    author_content = meta_auth.get("content", "") if meta_auth else ""
    data['author'] = str(author_content).strip() if author_content else "Unknown"
    meta_date = soup.find("meta", property="article:published_time")
    date_content = meta_date.get("content", "") if meta_date else ""
    data['date'] = str(date_content).strip() if date_content else None
    data['tags'] = [
        str(t.get("content")).strip()
        for t in soup.find_all("meta", property="article:tag")
        if t.get("content")
    ]

    # 3. Locate main content container
    # Priority ordered selectors for various site versions
    selectors = [
        ".tdb_single_content .tdb-block-inner",
        ".tdb_single_content",
        ".entry-content",
        ".post-content",
        "article",
        ".tdb-block-inner",
        ".main-content"
    ]

    container = None
    for sel in selectors:
        container = soup.select_one(sel)
        if container:
            break

    if not container:
        data['content'] = ""
        return data

    # 4. Remove technical and UI junk
    junk_tags = [
        "script", "style", "iframe", "noscript", "svg", "form", "button", 
        "input", ".td-social-sharing-buttons", ".share", ".navigation", 
        ".menu", ".sidebar", ".comments-area", ".related-posts", ".widget",
        ".advertisement", ".ads", ".banner"
    ]
    for junk in container.select(', '.join(junk_tags)):
        junk.decompose()

    # 5. Remove ad-hoc promotional elements based on attributes
    ad_keywords = ["bitco-", "ad-", "sponsor", "promo", "subscribe", "newsletter"]
    for el in container.find_all(["div", "section", "aside"]):
        # Skip malformed elements that have no attrs (e.g., stray </>)
        if not getattr(el, "attrs", None):
            continue

        el_id = str(el.get("id", "")).lower()
        el_cls = "".join(str(c).lower() for c in el.get("class") or [])
        el_style = str(el.get("style", "")).lower()

        if (
            any(k in el_id for k in ad_keywords)
            or any(k in el_cls for k in ad_keywords)
            or "display:none" in el_style
        ):
            el.decompose()

    # 6. Intelligent paragraph extraction
    # Recursive function to handle nested block elements correctly
    paragraphs = []
    block_tags = {
        'p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'blockquote', 
        'div', 'section', 'article', 'li'
    }

    def process_node(node, buffer: List[str]):
        if isinstance(node, NavigableString):
            text = str(node).strip()
            if text:
                buffer.append(text)
            return

        if not hasattr(node, 'name'):
            return

        name = node.name.lower()

        # If block element, flush buffer to new paragraph
        if name in block_tags or name == 'br':
            if buffer:
                text = ' '.join(buffer).strip()
                if len(text) >= 15:
                    paragraphs.append(text)
                buffer.clear()
            
            # Recurse into children
            if name != 'br':
                for child in node.children:
                    process_node(child, buffer)
                
                # Flush again after block ends
                if buffer:
                    text = ' '.join(buffer).strip()
                    if len(text) >= 15:
                        paragraphs.append(text)
                    buffer.clear()
        else:
            # Inline elements (span, b, i, a) - keep adding to buffer
            for child in node.children:
                process_node(child, buffer)

    current_buffer = []
    for child in container.children:
        process_node(child, current_buffer)
    
    # Flush remaining buffer
    if current_buffer:
        text = ' '.join(current_buffer).strip()
        if len(text) >= 15:
            paragraphs.append(text)

    # 7. Post-processing and cleaning
    cleaned_paras = []
    skip_phrases = [
        "share this", "follow us", "click here", "read more", 
        "rights reserved", "tags:", "categories:"
    ]

    for p in paragraphs:
        p_lower = p.lower()
        
        # Skip junk phrases
        if any(s in p_lower for s in skip_phrases):
            continue
            
        # Clean whitespace and punctuation
        clean = re.sub(r'\s+', ' ', p).strip()
        clean = re.sub(r'\s*([.,;:!?])', r'\1', clean) # Remove space before punct
        clean = re.sub(r'(?<=[.,;:!?])(?=[^\s])', r' ', clean) # Ensure space after punct
        
        cleaned_paras.append(clean)

    data['content'] = '\n'.join(cleaned_paras) if cleaned_paras else ""
    return data

In [7]:
def fetch_data(row):
    """Fetch and extract article data safely, handling errors gracefully."""
    HTML_DIR = f"{NEWS_DIR}/html"
    
    try:
        # Direct Series access with proper error handling
        timestamp = row['timestamp']
        article_id = row['id']

        # Check if we got valid data
        if timestamp is None or article_id is None:
            print(f"[!] Missing required data: timestamp={timestamp}, id={article_id}")
            return pd.Series({})

        if pd.isnull(timestamp):
            raise ValueError("Timestamp is NaT")

        # Format timestamp to string (e.g., 20251028_060616), ignoring timezone
        ts_str = timestamp.strftime("%Y%m%d_%H%M%S")

        file_name = f"{int(article_id):06d}_{ts_str}.html"
        file_path = os.path.join(HTML_DIR, file_name)

        # Check if file exists before processing
        if not os.path.exists(file_path):
            print(f"[!] HTML file not found: {file_path}")
            return pd.Series({})

        # Call your extraction logic (extract_article works correctly!)
        result = extract_article(file_path)
        return pd.Series(result)

    except Exception as e:
        # Get ID safely for error reporting
        try:
            article_id = row['id']
        except Exception as e2:
            article_id = "Unknown"
            print(f"[!] Warning - could not get article ID: {e2}")
        print(f"[!] Error processing ID {article_id}: {e}")
        # Return empty series to keep DataFrame structure aligned
        return pd.Series({})

#### Sentiment analysis

In [8]:
def mask_key(key: str, head: int = 5, tail: int = 4) -> str:
    """Mask the middle portion of a key using replace as requested."""
    if not key or len(key) <= head + tail:
        return "****"
    return key.replace(key[head:-tail], "****")


def init_cryptobert_classifier(
    hf_key: str,
    model_name: str,
    batch_size_gpu: int = 32,
    batch_size_cpu: int = 8,
    device: int | None = None,
):
    """Initialize CryptoBERT tokenizer and classifier with configurable batch sizes.
    Returns (tokenizer, classify_batch_fn).
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    resolved_device = device if device is not None else (0 if torch.cuda.is_available() else -1)

    if resolved_device >= 0:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            torch_dtype=(
                torch.bfloat16
                if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
                else torch.float16
            ),
        )
        classifier = pipeline(
            "text-classification",
            model=model,
            tokenizer=tokenizer,
            device=resolved_device,
            top_k=None,
        )
        default_batch_size = batch_size_gpu

        def _classify_batch(texts, batch_size=None):
            effective_batch = batch_size or default_batch_size
            outs = classifier(texts, batch_size=effective_batch)
            res = []
            for out in outs:
                score_map = {o["label"]: o["score"] for o in out}
                res.append([
                    score_map.get("Bullish", 0.0),
                    score_map.get("Neutral", 0.0),
                    score_map.get("Bearish", 0.0),
                ])
            return res
    else:
        client = InferenceClient(
            provider="auto",
            api_key=hf_key,
        )
        default_batch_size = batch_size_cpu

        def _classify_batch(texts, batch_size=None):
            res = []
            for text in texts:
                result = client.text_classification(text=text, model=model_name)
                score_map = {}
                for x in result:
                    label = getattr(x, "label", None) or x["label"]
                    score = getattr(x, "score", None) or x["score"]
                    score_map[label] = score
                res.append([
                    score_map.get("Bullish", 0.0),
                    score_map.get("Neutral", 0.0),
                    score_map.get("Bearish", 0.0),
                ])
            return res

    return tokenizer, _classify_batch


def chunk_by_tokens(
    text: str,
    tokenizer,
    chunk_size: int = 256,
    overlap: int = 48,
    max_chunks: int = 10,
):
    """Split text into token-based chunks with overlap."""
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    start = 0
    step = chunk_size - overlap

    while start < len(tokens) and len(chunks) < max_chunks:
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunks.append(tokenizer.decode(chunk_tokens))
        start += step

    return chunks


def cryptobert_sentiment_long_article(
    title: str,
    content: str,
    tokenizer,
    classify_batch,
    chunk_size: int = 256,
    overlap: int = 48,
    max_chunks: int = 10,
    topk: int = 3,
    head_char_limit: int = 1000,
):
    """Compute sentiment for a long article with head-focused and global aggregation."""
    title = title or ""
    content = content or ""

    full_text = f"{title}\n\n{content}"

    # 1) HEAD: title + first ~1000 characters
    head_text = f"{title}\n\n{content[:head_char_limit]}"
    head_chunks = chunk_by_tokens(
        head_text,
        tokenizer,
        chunk_size=chunk_size,
        overlap=overlap,
        max_chunks=max_chunks,
    )

    head_scores = (
        np.array(classify_batch(head_chunks))
        if head_chunks
        else np.zeros((1, 3))
    )
    head_mean = head_scores.mean(axis=0)

    # 2) Global view: entire article
    all_chunks = chunk_by_tokens(
        full_text,
        tokenizer,
        chunk_size=chunk_size,
        overlap=overlap,
        max_chunks=max_chunks,
    )

    all_scores = (
        np.array(classify_batch(all_chunks))
        if all_chunks
        else np.zeros((1, 3))
    )

    global_mean = all_scores.mean(axis=0)
    global_max = all_scores.max(axis=0)

    k = min(topk, all_scores.shape[0])
    if k > 0:
        idx_bull = np.argsort(-all_scores[:, 0])[:k]
        idx_bear = np.argsort(-all_scores[:, 2])[:k]
        topk_bull_mean = all_scores[idx_bull, 0].mean()
        topk_bear_mean = all_scores[idx_bear, 2].mean()
    else:
        topk_bull_mean = 0.0
        topk_bear_mean = 0.0

    return {
        "head_p_bull": float(head_mean[0]),
        "head_p_neu":  float(head_mean[1]),
        "head_p_bear": float(head_mean[2]),

        "mean_p_bull": float(global_mean[0]),
        "mean_p_neu":  float(global_mean[1]),
        "mean_p_bear": float(global_mean[2]),

        "max_p_bull": float(global_max[0]),
        "max_p_neu":  float(global_max[1]),
        "max_p_bear": float(global_max[2]),

        "topk_mean_p_bull": float(topk_bull_mean),
        "topk_mean_p_bear": float(topk_bear_mean),

        "head_sent_net": float(head_mean[0] - head_mean[2]),
        "global_sent_net": float(global_mean[0] - global_mean[2]),
    }


#### Merge news

In [9]:
def map_to_actionable_time(timestamp, timeframe_str="4h"):
    """
    Map timestamp to AS-OF actionable candle open time (NO FUTURE LEAKAGE).
    News in (T-Δ, T] -> mapped to candle T.
    """
    if pd.isnull(timestamp):
        return None
    
    if isinstance(timestamp, str):
        timestamp = pd.to_datetime(timestamp)

    match = re.match(r"(\d+)?([a-zA-Z]+)", timeframe_str)
    if not match:
        raise ValueError(f"Invalid timeframe: {timeframe_str}")
    
    amount, unit_raw = match.groups()
    amount = int(amount or 1)
    unit_raw = unit_raw.lower()

    try:
        if unit_raw in ['h', 'hour']:
            freq = f"{amount}h"
            floored = timestamp.floor(freq)

            # AS-OF LOGIC:
            # if news exactly at open -> belongs to this candle
            # else -> belongs to NEXT open (i.e., as-of bucket)
            if timestamp == floored:
                return floored
            else:
                return floored + pd.Timedelta(hours=amount)

        elif unit_raw in ['d', 'day']:
            floored = timestamp.floor("1D")
            if timestamp == floored:
                return floored
            else:
                return floored + pd.Timedelta(days=amount)

        elif unit_raw in ['w', 'week']:
            start_of_week = (timestamp - pd.Timedelta(days=timestamp.weekday())).normalize()
            if timestamp == start_of_week:
                return start_of_week
            else:
                return start_of_week + pd.Timedelta(weeks=amount)

        elif unit_raw in ['m', 'mo', 'month']:
            start_of_month = timestamp.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
            if timestamp == start_of_month:
                return start_of_month
            else:
                return start_of_month + pd.DateOffset(months=amount)

        else:
            raise ValueError(f"Unsupported unit: {unit_raw}")

    except Exception as e:
        print(f"[!] Error mapping time {timestamp}: {e}")
        return None

In [10]:
def format_merged_content(group):
    """
    Helper to merge multiple articles into a single structured string.
    """
    merged_text = ""
    group = group.sort_values('timestamp')
    
    for _, row in group.iterrows():
        title = row.get('title', 'No Title')
        content = row.get('content', '')
        
        merged_text += f"# [Article ID: {row['id']}] {title}\n"
        merged_text += f"{content}\n"
        merged_text += "---\n"
        
    return merged_text.strip()


In [11]:
def process_and_merge_news(df: pd.DataFrame, timeframe: str = '4h') -> pd.DataFrame:
    print(f"Grouping news by AS-OF timeframe: {timeframe}...")

    df = df.copy()

    # Sentiment parameters
    K_ALPHA = 2.0
    EXAG_HEAD_THR = 0.5
    EXAG_GLOBAL_THR = 0.2
    BULL_THR = 0.1
    BEAR_THR = -0.1
    STRONG_BULL_THR = 0.3
    STRONG_BEAR_THR = -0.3

    # Ensure timestamp is datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

    # Map to actionable_time
    df['actionable_time'] = df['timestamp'].apply(
        lambda x: map_to_actionable_time(x, timeframe)
    )
    df_clean = df.dropna(subset=['actionable_time']).copy()

    # Compute per-news features
    head = df_clean['head_sent_net']
    glob = df_clean['global_sent_net']

    alpha = 1.0 / (1.0 + K_ALPHA * (head - glob).abs())
    df_clean['combined_sent'] = alpha * glob + (1 - alpha) * head

    df_clean['head_global_diff'] = head - glob
    df_clean['head_global_abs_diff'] = (head - glob).abs()

    df_clean['is_head_exag'] = (
        (head.abs() > EXAG_HEAD_THR) &
        (glob.abs() < EXAG_GLOBAL_THR)
    ).astype(int)

    # Aggregation per actionable_time
    def _agg_bucket(x: pd.DataFrame) -> pd.Series:
        n = len(x)

        # CASE A: NO NEWS
        if n == 0:
            return pd.Series({
                'merged_content': None,
                'news_article_count': 0,
                'news_log_article_count': 0.0,
                'has_news': 0,
                'news_combined_sent_mean': np.nan,
                'news_combined_sent_std': np.nan,
                'news_combined_sent_max': np.nan,
                'news_combined_sent_min': np.nan,
                'news_combined_sent_max_abs': np.nan,
                'news_combined_sent_sum_abs': np.nan,
                'news_bull_ratio': np.nan,
                'news_bear_ratio': np.nan,
                'news_p_bull_mean': np.nan,
                'news_p_bear_mean': np.nan,
                'news_p_neu_mean': np.nan,
                'news_max_p_bull_bucket': np.nan,
                'news_max_p_bear_bucket': np.nan,
                'news_is_conflict': np.nan,
                'news_head_sent_mean': np.nan,
                'news_head_sent_std': np.nan,
                'news_head_sent_max': np.nan,
                'news_head_sent_min': np.nan,
                'news_head_global_diff_mean': np.nan,
                'news_head_global_abs_diff_mean': np.nan,
                'news_head_exag_ratio': np.nan,
                'original_ids': []
            })

        # CASE B: HAVE NEWS >= 1
        combined = x['combined_sent']
        head_sent = x['head_sent_net']

        # Bull/Bear masks
        bull_mask = combined > BULL_THR
        bear_mask = combined < BEAR_THR

        strong_bull_mask = combined > STRONG_BULL_THR
        strong_bear_mask = combined < STRONG_BEAR_THR

        # Compute std correctly - if only 1 valid news: std = 0
        combined_non_null = combined.dropna()
        if len(combined_non_null) <= 1:
            combined_std = 0.0
        else:
            combined_std = combined_non_null.std()

        head_non_null = head_sent.dropna()
        if len(head_non_null) <= 1:
            head_std = 0.0
        else:
            head_std = head_non_null.std()

        return pd.Series({

            # TEXT
            'merged_content': format_merged_content(x),

            # VOLUME
            'news_article_count': n,
            'news_log_article_count': np.log1p(n),
            'has_news': 1,

            # SENTIMENT COMBINED
            'news_combined_sent_mean': combined_non_null.mean(),
            'news_combined_sent_std': combined_std,
            'news_combined_sent_max': combined_non_null.max(),
            'news_combined_sent_min': combined_non_null.min(),
            'news_combined_sent_max_abs': np.abs(combined_non_null).max(),
            'news_combined_sent_sum_abs': np.abs(combined_non_null).sum(),

            # BULL/BEAR STRUCTURE
            'news_bull_ratio': bull_mask.mean(),
            'news_bear_ratio': bear_mask.mean(),

            # CRYPTOBERT PROBS
            'news_p_bull_mean': x['mean_p_bull'].mean(),
            'news_p_bear_mean': x['mean_p_bear'].mean(),
            'news_p_neu_mean': x['mean_p_neu'].mean(),
            'news_max_p_bull_bucket': x['max_p_bull'].max(),
            'news_max_p_bear_bucket': x['max_p_bear'].max(),

            # CONFLICT
            'news_is_conflict': int(strong_bull_mask.any() and strong_bear_mask.any()),

            # HEAD SENTIMENT
            'news_head_sent_mean': head_non_null.mean(),
            'news_head_sent_std': head_std,
            'news_head_sent_max': head_non_null.max(),
            'news_head_sent_min': head_non_null.min(),

            # MISMATCH
            'news_head_global_diff_mean': x['head_global_diff'].mean(),
            'news_head_global_abs_diff_mean': x['head_global_abs_diff'].mean(),
            'news_head_exag_ratio': x['is_head_exag'].mean(),
        })

    # Group apply
    grouped = df_clean.groupby('actionable_time').apply(_agg_bucket)

    grouped = grouped.sort_index().reset_index()

    return grouped

## Build Dataset

### Load

In [12]:
def load_data(label, path):
    """Load CSV file and return DataFrame with info display"""
    if os.path.exists(path):
        df = pd.read_csv(path)
        
        # Handle different timestamp column names
        timestamp_col = 'timestamp'  # default
        if timestamp_col not in df.columns:
            # Try common alternatives
            for col in ['datetime', 'time', 'date']:
                if col in df.columns:
                    timestamp_col = col
                    break
        
        if timestamp_col and timestamp_col in df.columns:
            print(f"{label}: {len(df)} records ({df[timestamp_col].min()} → {df[timestamp_col].max()})")
        else:
            print(f"{label}: {len(df)} records (no timestamp column found)")
        
        return df
    else:
        print(f"✗ {label}: File not found")
        return None

# Data sources configuration
DATA_SOURCES = [
    ("OHLCV", COINS, get_ohlcv_file),
    ("Market Cap", None, get_marketcap_file),
    ("Network Activity", COINS, get_networkactivity_file),
    ("Mining", None, get_mining_file),
    ("Onchain Metrics", ["BTC"], get_onchainmetrics_file),
    ("News", None, get_news_file),
    ("Sentiment Index", None, get_sentimentindex_file),
]

# Load all data sources into DataFrames
dataframes = {}

for source_name, coins, get_file_func in DATA_SOURCES:
    if coins:
        # Multiple coins for this source type
        for coin in coins:
            label = f"{coin} {source_name}"
            df = load_data(label, get_file_func(coin))
            if df is not None:
                # Store with clean key name
                key = f"{coin.lower()}_{source_name.lower().replace(' ', '_')}"
                dataframes[key] = df
    else:
        # Single file for this source type
        df = load_data(source_name, get_file_func())
        if df is not None:
            # Store with clean key name
            key = source_name.lower().replace(' ', '_')
            dataframes[key] = df
    print("-"*10)

print(f"\n✓ Loaded {len(dataframes)} DataFrames:")
print(list(dataframes.keys()))

BTC OHLCV: 18050 records (2017-08-17 04:00:00+00:00 → 2025-11-14 00:00:00+00:00)
ETH OHLCV: 18050 records (2017-08-17 04:00:00+00:00 → 2025-11-14 00:00:00+00:00)
----------
Market Cap: 17129 records (2013-01-01 03:59:59.999000+00:00 → 2025-11-13 23:59:59.999000+00:00)
----------
BTC Network Activity: 6160 records (2009-01-03 00:00:00+00:00 → 2025-11-14 00:00:00+00:00)
ETH Network Activity: 3761 records (2015-07-30 00:00:00+00:00 → 2025-11-14 00:00:00+00:00)
----------
Mining: 3077 records (2009-01-03 00:00:00+00:00 → 2025-11-13 00:00:00+00:00)
----------
BTC Onchain Metrics: 6160 records (2009-01-03 00:00:00+00:00 → 2025-11-14 00:00:00+00:00)
----------
News: 13388 records (2012-02-28T06:06:16+00:00 → 2025-11-13T18:38:33.700701+00:00)
----------
Sentiment Index: 2840 records (2018-02-01 00:00:00+00:00 → 2025-11-14 00:00:00+00:00)
----------

✓ Loaded 9 DataFrames:
['btc_ohlcv', 'eth_ohlcv', 'market_cap', 'btc_network_activity', 'eth_network_activity', 'mining', 'btc_onchain_metrics', '

#### OHLCV

##### BTC

In [13]:
dataframes["btc_ohlcv"].head(5)

,timestamp,open,high,low,close,volume,quote_volume,trades
0,2017-08-17 04:00:00+00:00,4261.48,4349.99,4261.32,4349.99,82.088865,3.531943e+05,334
1,2017-08-17 08:00:00+00:00,4333.32,4485.39,4333.32,4427.30,63.619882,2.825012e+05,248
2,2017-08-17 12:00:00+00:00,4436.06,4485.39,4333.42,4352.34,174.562001,7.742388e+05,858
3,2017-08-17 16:00:00+00:00,4352.33,4354.84,4200.74,4325.23,225.109716,9.652911e+05,986
4,2017-08-17 20:00:00+00:00,4307.56,4369.69,4258.56,4285.08,249.769913,1.079545e+06,1001


In [14]:
dataframes["btc_ohlcv"] = convert_timestamp(dataframes["btc_ohlcv"], "timestamp")
dataframes["btc_ohlcv"].rename(columns=lambda x: f"BTC_{x}" if x != "timestamp" else x, inplace=True)
dataframes["btc_ohlcv"].head(5)

,timestamp,BTC_open,BTC_high,BTC_low,BTC_close,BTC_volume,BTC_quote_volume,BTC_trades
0,2017-08-17 04:00:00+00:00,4261.48,4349.99,4261.32,4349.99,82.088865,3.531943e+05,334
1,2017-08-17 08:00:00+00:00,4333.32,4485.39,4333.32,4427.30,63.619882,2.825012e+05,248
2,2017-08-17 12:00:00+00:00,4436.06,4485.39,4333.42,4352.34,174.562001,7.742388e+05,858
3,2017-08-17 16:00:00+00:00,4352.33,4354.84,4200.74,4325.23,225.109716,9.652911e+05,986
4,2017-08-17 20:00:00+00:00,4307.56,4369.69,4258.56,4285.08,249.769913,1.079545e+06,1001


##### ETH

In [15]:
dataframes["eth_ohlcv"].head(5)

,timestamp,open,high,low,close,volume,quote_volume,trades
0,2017-08-17 04:00:00+00:00,301.13,307.96,298.00,307.96,1561.95305,473487.665119,711
1,2017-08-17 08:00:00+00:00,307.95,312.00,307.00,308.95,1177.71088,364545.316402,775
2,2017-08-17 12:00:00+00:00,308.95,310.51,303.56,307.06,1882.05267,578644.931890,1140
3,2017-08-17 16:00:00+00:00,307.74,312.18,298.21,301.60,1208.05192,370209.051467,957
4,2017-08-17 20:00:00+00:00,301.60,310.85,299.01,302.00,1200.94182,367768.335479,939


In [16]:
dataframes["eth_ohlcv"] = convert_timestamp(dataframes["eth_ohlcv"], "timestamp")
dataframes["eth_ohlcv"].rename(columns=lambda x: f"ETH_{x}" if x != "timestamp" else x, inplace=True)
dataframes["eth_ohlcv"].head(5)

,timestamp,ETH_open,ETH_high,ETH_low,ETH_close,ETH_volume,ETH_quote_volume,ETH_trades
0,2017-08-17 04:00:00+00:00,301.13,307.96,298.00,307.96,1561.95305,473487.665119,711
1,2017-08-17 08:00:00+00:00,307.95,312.00,307.00,308.95,1177.71088,364545.316402,775
2,2017-08-17 12:00:00+00:00,308.95,310.51,303.56,307.06,1882.05267,578644.931890,1140
3,2017-08-17 16:00:00+00:00,307.74,312.18,298.21,301.60,1208.05192,370209.051467,957
4,2017-08-17 20:00:00+00:00,301.60,310.85,299.01,302.00,1200.94182,367768.335479,939


#### Marketcap

In [17]:
dataframes["market_cap"].head(5)

,timestamp,BTC_market_cap,ETH_market_cap,USDT_market_cap,USDC_market_cap
0,2013-01-01 03:59:59.999000+00:00,1.571684e+08,NaN,NaN,NaN
1,2013-01-01 15:59:59.999000+00:00,1.497462e+08,NaN,NaN,NaN
2,2013-01-02 03:59:59.999000+00:00,1.424379e+08,NaN,NaN,NaN
3,2013-01-02 15:59:59.999000+00:00,1.492968e+08,NaN,NaN,NaN
4,2013-01-03 03:59:59.999000+00:00,1.372253e+08,NaN,NaN,NaN


In [18]:
dataframes["market_cap"] = convert_timestamp(dataframes["market_cap"], "timestamp", round_to="h")
dataframes["market_cap"].head(5)

,timestamp,BTC_market_cap,ETH_market_cap,USDT_market_cap,USDC_market_cap
0,2013-01-01 04:00:00+00:00,1.571684e+08,NaN,NaN,NaN
1,2013-01-01 16:00:00+00:00,1.497462e+08,NaN,NaN,NaN
2,2013-01-02 04:00:00+00:00,1.424379e+08,NaN,NaN,NaN
3,2013-01-02 16:00:00+00:00,1.492968e+08,NaN,NaN,NaN
4,2013-01-03 04:00:00+00:00,1.372253e+08,NaN,NaN,NaN


#### Network activity

##### BTC

In [19]:
dataframes["btc_network_activity"].head(5)

,timestamp,active_addresses,tx_count
0,2009-01-03 00:00:00+00:00,0.0,0.0
1,2009-01-04 00:00:00+00:00,0.0,0.0
2,2009-01-05 00:00:00+00:00,0.0,0.0
3,2009-01-06 00:00:00+00:00,0.0,0.0
4,2009-01-07 00:00:00+00:00,0.0,0.0


In [20]:
dataframes["btc_network_activity"].rename(columns=lambda x: f"BTC_{x}" if x != "timestamp" else x, inplace=True)
dataframes["btc_network_activity"].head(5)

,timestamp,BTC_active_addresses,BTC_tx_count
0,2009-01-03 00:00:00+00:00,0.0,0.0
1,2009-01-04 00:00:00+00:00,0.0,0.0
2,2009-01-05 00:00:00+00:00,0.0,0.0
3,2009-01-06 00:00:00+00:00,0.0,0.0
4,2009-01-07 00:00:00+00:00,0.0,0.0


##### ETH

In [21]:
dataframes['eth_network_activity'].head(5)

,timestamp,active_addresses,tx_count
0,2015-07-30 00:00:00+00:00,9206.0,0.0
1,2015-07-31 00:00:00+00:00,424.0,0.0
2,2015-08-01 00:00:00+00:00,413.0,0.0
3,2015-08-02 00:00:00+00:00,432.0,0.0
4,2015-08-03 00:00:00+00:00,444.0,0.0


In [22]:
dataframes["eth_network_activity"].rename(columns=lambda x: f"ETH_{x}" if x != "timestamp" else x, inplace=True)
dataframes["eth_network_activity"].head(5)

,timestamp,ETH_active_addresses,ETH_tx_count
0,2015-07-30 00:00:00+00:00,9206.0,0.0
1,2015-07-31 00:00:00+00:00,424.0,0.0
2,2015-08-01 00:00:00+00:00,413.0,0.0
3,2015-08-02 00:00:00+00:00,432.0,0.0
4,2015-08-03 00:00:00+00:00,444.0,0.0


#### Mining

In [23]:
dataframes["mining"].head(5)

,timestamp,mining_difficulty,hash_rate_ths,miner_revenue_usd
0,2009-01-03 00:00:00+00:00,1.0,4.971027e-08,0.0
1,2009-01-07 00:00:00+00:00,0.0,0.000000e+00,0.0
2,2009-01-11 00:00:00+00:00,1.0,5.269289e-06,0.0
3,2009-01-15 00:00:00+00:00,1.0,6.313204e-06,0.0
4,2009-01-17 00:00:00+00:00,1.0,6.313204e-06,0.0


In [24]:
dataframes["mining"] = convert_timestamp(dataframes["mining"], "timestamp")
dataframes["mining"].head(5)

,timestamp,mining_difficulty,hash_rate_ths,miner_revenue_usd
0,2009-01-03 00:00:00+00:00,1.0,4.971027e-08,0.0
1,2009-01-07 00:00:00+00:00,0.0,0.000000e+00,0.0
2,2009-01-11 00:00:00+00:00,1.0,5.269289e-06,0.0
3,2009-01-15 00:00:00+00:00,1.0,6.313204e-06,0.0
4,2009-01-17 00:00:00+00:00,1.0,6.313204e-06,0.0


#### Onchain metrics

In [25]:
dataframes["btc_onchain_metrics"].head(5)

,timestamp,total_supply,mvrv_ratio,exchange_inflow_native,exchange_inflow_usd,exchange_outflow_native,exchange_outflow_usd,exchange_supply_native,exchange_supply_usd
0,2009-01-03 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2009-01-04 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2009-01-05 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2009-01-06 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2009-01-07 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
dataframes["btc_onchain_metrics"].rename(columns=lambda x: f"BTC_onchain_{x}" if x != "timestamp" else x, inplace=True)
dataframes["btc_onchain_metrics"].head(5)

,timestamp,BTC_onchain_total_supply,BTC_onchain_mvrv_ratio,BTC_onchain_exchange_inflow_native,BTC_onchain_exchange_inflow_usd,BTC_onchain_exchange_outflow_native,BTC_onchain_exchange_outflow_usd,BTC_onchain_exchange_supply_native,BTC_onchain_exchange_supply_usd
0,2009-01-03 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2009-01-04 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2009-01-05 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2009-01-06 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2009-01-07 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### News

In [27]:
pd.concat([
    dataframes['news'].head(5),
    dataframes['news'].tail(5)
])

,id,datetime,url,status
0,1,2012-02-28T06:06:16+00:00,https://bitcoinmagazine.com/business/the-waste...,1
1,2,2012-02-28T11:34:40+00:00,https://bitcoinmagazine.com/culture/bitcoin-ad...,1
2,3,2012-02-28T22:46:22+00:00,https://bitcoinmagazine.com/technical/bitcoin-...,1
3,4,2012-02-28T23:24:16+00:00,https://bitcoinmagazine.com/technical/client-s...,1
4,5,2012-02-29T03:24:31+00:00,https://bitcoinmagazine.com/culture/traditiona...,1
13383,13384,2025-11-13T18:38:11.107088+00:00,https://bitcoinmagazine.com/news/china-accuses...,1
13384,13385,2025-11-13T18:38:16.783807+00:00,https://bitcoinmagazine.com/markets/bitcoin-pr...,1
13385,13386,2025-11-13T18:38:22.411648+00:00,https://bitcoinmagazine.com/business/building-...,1
13386,13387,2025-11-13T18:38:28.092940+00:00,https://bitcoinmagazine.com/news/sofi-makes-hi...,1
13387,13388,2025-11-13T18:38:33.700701+00:00,https://bitcoinmagazine.com/markets/bitcoin-pr...,1


In [28]:
dataframes['news'] = dataframes['news'].rename(columns={"datetime": "timestamp"})
dataframes['news'] = convert_timestamp(dataframes['news'], 'timestamp', round_to="s")
pd.concat([
    dataframes['news'].head(5),
    dataframes['news'].tail(5)
])

,id,timestamp,url,status
0,1,2012-02-28 06:06:16+00:00,https://bitcoinmagazine.com/business/the-waste...,1
1,2,2012-02-28 11:34:40+00:00,https://bitcoinmagazine.com/culture/bitcoin-ad...,1
2,3,2012-02-28 22:46:22+00:00,https://bitcoinmagazine.com/technical/bitcoin-...,1
3,4,2012-02-28 23:24:16+00:00,https://bitcoinmagazine.com/technical/client-s...,1
4,5,2012-02-29 03:24:31+00:00,https://bitcoinmagazine.com/culture/traditiona...,1
13383,13384,2025-11-13 18:38:11+00:00,https://bitcoinmagazine.com/news/china-accuses...,1
13384,13385,2025-11-13 18:38:16+00:00,https://bitcoinmagazine.com/markets/bitcoin-pr...,1
13385,13386,2025-11-13 18:38:22+00:00,https://bitcoinmagazine.com/business/building-...,1
13386,13387,2025-11-13 18:38:28+00:00,https://bitcoinmagazine.com/news/sofi-makes-hi...,1
13387,13388,2025-11-13 18:38:33+00:00,https://bitcoinmagazine.com/markets/bitcoin-pr...,1


In [29]:
# Try to load prior extracted crawl (if it exists) and merge on id
try:
    prev = pd.read_csv(f"{NEWS_DIR}/bitcoinmagazinenews_crawl.csv", sep="\t")
    # Keep only extraction columns you trust
    extract_cols = ["title", "content", "author", "date", "tags"]
    prev = prev[["id"] + [c for c in extract_cols if c in prev.columns]]
    news = dataframes["news"].merge(prev, on="id", how="left", suffixes=("", "_old"))
except FileNotFoundError:
    news = dataframes["news"].copy()

# Determine which rows actually need fetching
need_cols = ["title", "content"]
present_cols = [c for c in need_cols if c in news.columns]
if present_cols:
    mask_missing = (
        news[present_cols].isna().any(axis=1)
        | news["content"].fillna("").str.strip().eq("")
    )
else:
    # If neither column exists, only then fetch all
    mask_missing = pd.Series(True, index=news.index)

if mask_missing.any():
    fetched = news.loc[mask_missing].progress_apply(fetch_data, axis=1)
    cols = fetched.columns
    news.loc[mask_missing, cols] = news.loc[mask_missing, cols].combine_first(fetched)

dataframes["news"] = news

  0%|          | 0/60 [00:00<?, ?it/s]

In [30]:
pd.concat([
    dataframes['news'].head(2),
    dataframes['news'].tail(2)
])

,id,timestamp,url,status,title,content,author,date,tags
0,1,2012-02-28 06:06:16+00:00,https://bitcoinmagazine.com/business/the-waste...,1,The Wasted Electricity Objection To Bitcoin,One of the main arguments in favor of fiat cur...,Vitalik Buterin,2012-02-28T06:06:16-05:00,"['energy consumption', 'energy waste', 'Fees',..."
1,2,2012-02-28 11:34:40+00:00,https://bitcoinmagazine.com/culture/bitcoin-ad...,1,Bitcoin Adoption Opportunity: Teenagers,If Bitcoin is to achieve mainstream success it...,Vitalik Buterin,2012-02-28T11:34:40-05:00,"['Cards', 'Internet', 'People']"
13386,13387,2025-11-13 18:38:28+00:00,https://bitcoinmagazine.com/news/sofi-makes-hi...,1,SoFi Enters The Bitcoin Era As First U.S. Bank...,SoFi Technologies (NASDAQ: SOFI) has become th...,Micah Zimmerman,2025-11-11T09:52:03-05:00,"['Banking', 'Bitcoin', 'Bitcoin Trading', 'Ret..."
13387,13388,2025-11-13 18:38:33+00:00,https://bitcoinmagazine.com/markets/bitcoin-pr...,1,Bitcoin Price Outlook For Reaching $1 Million ...,Bitcoin price long-term trajectory has been ex...,Matt Crosby,2025-11-11T09:29:33-05:00,"['Bitcoin', 'Bitcoin Magazine Pro', 'bitcoin p..."


In [31]:
# Find safe separator for CSV storage
candidates = [",", ";", "|", "\t", "~", "^"]

df_str = dataframes['news'].astype(str)

safe_seps = []
bad_seps = []

for sep in candidates:
    exists = df_str.apply(lambda col: col.str.contains(sep, regex=False)).any().any()
    if exists:
        bad_seps.append(sep)
    else:
        safe_seps.append(sep)

print("Safe separators:", safe_seps)
print("Found in data:", bad_seps)

Safe separators: ['\t']
Found in data: [',', ';', '|', '~', '^']


In [32]:
dataframes['news'].to_csv(f"{NEWS_DIR}/bitcoinmagazinenews_crawl.csv", sep='\t', index=False)
df_news = pd.read_csv(f"{NEWS_DIR}/bitcoinmagazinenews_crawl.csv", sep='\t')
pd.concat([
    df_news.head(1),
    df_news.tail(1)
])

,id,timestamp,url,status,title,content,author,date,tags
0,1,2012-02-28 06:06:16+00:00,https://bitcoinmagazine.com/business/the-waste...,1,The Wasted Electricity Objection To Bitcoin,One of the main arguments in favor of fiat cur...,Vitalik Buterin,2012-02-28T06:06:16-05:00,"['energy consumption', 'energy waste', 'Fees',..."
13387,13388,2025-11-13 18:38:33+00:00,https://bitcoinmagazine.com/markets/bitcoin-pr...,1,Bitcoin Price Outlook For Reaching $1 Million ...,Bitcoin price long-term trajectory has been ex...,Matt Crosby,2025-11-11T09:29:33-05:00,"['Bitcoin', 'Bitcoin Magazine Pro', 'bitcoin p..."


In [33]:
# Check for missing content or titles
nan_rows = df_news[df_news["content"].isna() | df_news["title"].isna()]
print(f"Rows with missing content/title: {len(nan_rows)} ({round(len(nan_rows)/len(df_news)*100, 2)}%)")

Rows with missing content/title: 60 (0.45%)


In [34]:
# Drop rows with missing content or title
df_news = df_news.dropna(subset=["content", "title"])
new_nan_rows = df_news[df_news["content"].isna() | df_news["title"].isna()]
print(f"Remaining rows with missing data: {len(new_nan_rows)}")

Remaining rows with missing data: 0


In [35]:
SENTIMENT_CONFIG = {
    "model_name": "ElKulako/cryptobert",
    "chunk_size": 256,
    "overlap": 48,
    "max_chunks": 16,
    "topk": 3,
    "head_char_limit": 1000,
    "batch_size_gpu": 256,
    "batch_size_cpu": 8,
    "device": None,
}

sentiment_cols = [
    "head_p_bull", "head_p_neu", "head_p_bear",
    "mean_p_bull", "mean_p_neu", "mean_p_bear",
    "max_p_bull", "max_p_neu", "max_p_bear",
    "topk_mean_p_bull", "topk_mean_p_bear",
    "head_sent_net", "global_sent_net",
]

# Load previously-scored rows if available
try:
    prev = pd.read_csv(f"{NEWS_DIR}/bitcoinmagazinenews_extract.csv", sep="\t")
    keep_cols = ["id"] + [c for c in sentiment_cols if c in prev.columns]
    prev_scores = prev[keep_cols]
    df_news = df_news.merge(prev_scores, on="id", how="left", suffixes=("", "_old"))
    for c in sentiment_cols:
        if f"{c}_old" in df_news:
            df_news[c] = df_news[c].combine_first(df_news[f"{c}_old"])
            df_news.drop(columns=[f"{c}_old"], inplace=True)
except FileNotFoundError:
    pass

# Ensure sentiment columns exist
for c in sentiment_cols:
    if c not in df_news:
        df_news[c] = pd.NA

if hf_key is None:
    print("Warning: hf_key not found in environment variables")
else:
    tokenizer, classify_batch = init_cryptobert_classifier(
        hf_key=hf_key,
        model_name=SENTIMENT_CONFIG["model_name"],
        batch_size_gpu=SENTIMENT_CONFIG["batch_size_gpu"],
        batch_size_cpu=SENTIMENT_CONFIG["batch_size_cpu"],
        device=SENTIMENT_CONFIG["device"],
    )

    # Only score rows with missing sentiment AND with content available
    needs_score = df_news[sentiment_cols].isna().any(axis=1) & df_news["content"].notna()
    to_score = df_news.loc[needs_score, ["title", "content"]]

    scores = []
    row_ids = []
    for idx, (title, content) in tqdm(to_score.iterrows(), total=len(to_score)):
        scores.append(
            cryptobert_sentiment_long_article(
                title=title or "",
                content=content or "",
                tokenizer=tokenizer,
                classify_batch=classify_batch,
                chunk_size=SENTIMENT_CONFIG["chunk_size"],
                overlap=SENTIMENT_CONFIG["overlap"],
                max_chunks=SENTIMENT_CONFIG["max_chunks"],
                topk=SENTIMENT_CONFIG["topk"],
                head_char_limit=SENTIMENT_CONFIG["head_char_limit"],
            )
        )
        row_ids.append(idx)

    if scores:
        score_df = pd.DataFrame(scores, index=row_ids)
        df_news.loc[score_df.index, sentiment_cols] = score_df[sentiment_cols]

df_news.head(5)
df_news.to_csv(f"{NEWS_DIR}/bitcoinmagazinenews_extract.csv", sep="\t", index=False)

0it [00:00, ?it/s]

In [36]:
df_news = pd.read_csv(f"{NEWS_DIR}/bitcoinmagazinenews_extract.csv", sep='\t')
pd.concat([
    df_news.head(1),
    df_news.tail(1)
])

,id,timestamp,url,status,title,content,author,date,tags,head_p_bull,...,mean_p_bull,mean_p_neu,mean_p_bear,max_p_bull,max_p_neu,max_p_bear,topk_mean_p_bull,topk_mean_p_bear,head_sent_net,global_sent_net
0,1,2012-02-28 06:06:16+00:00,https://bitcoinmagazine.com/business/the-waste...,1,The Wasted Electricity Objection To Bitcoin,One of the main arguments in favor of fiat cur...,Vitalik Buterin,2012-02-28T06:06:16-05:00,"['energy consumption', 'energy waste', 'Fees',...",0.633942,...,0.350466,0.288996,0.360538,0.691526,0.595159,0.806509,0.512493,0.610582,0.500018,-0.010072
13327,13388,2025-11-13 18:38:33+00:00,https://bitcoinmagazine.com/markets/bitcoin-pr...,1,Bitcoin Price Outlook For Reaching $1 Million ...,Bitcoin price long-term trajectory has been ex...,Matt Crosby,2025-11-11T09:29:33-05:00,"['Bitcoin', 'Bitcoin Magazine Pro', 'bitcoin p...",0.427240,...,0.371594,0.285235,0.343171,0.577080,0.415655,0.728282,0.506699,0.557618,0.255401,0.028422


In [37]:
df_news_merged = process_and_merge_news(df_news, timeframe='4h')
pd.concat([
    df_news_merged.head(2),
    df_news_merged.tail(2)
])

Grouping news by AS-OF timeframe: 4h...


/var/folders/h3/4_pq_7pd3pq2286xkp78c1480000gn/T/ipykernel_96392/2959001170.py:143: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = df_clean.groupby('actionable_time').apply(_agg_bucket)


,actionable_time,merged_content,news_article_count,news_log_article_count,has_news,news_combined_sent_mean,news_combined_sent_std,news_combined_sent_max,news_combined_sent_min,news_combined_sent_max_abs,...,news_max_p_bull_bucket,news_max_p_bear_bucket,news_is_conflict,news_head_sent_mean,news_head_sent_std,news_head_sent_max,news_head_sent_min,news_head_global_diff_mean,news_head_global_abs_diff_mean,news_head_exag_ratio
0,2012-02-28 08:00:00+00:00,# [Article ID: 1] The Wasted Electricity Objec...,1,0.693147,1,0.247521,0.000000,0.247521,0.247521,0.247521,...,0.691526,0.806509,0,0.500018,0.000000,0.500018,0.500018,0.510090,0.510090,1.000000
1,2012-02-28 12:00:00+00:00,# [Article ID: 2] Bitcoin Adoption Opportunity...,1,0.693147,1,0.412206,0.000000,0.412206,0.412206,0.412206,...,0.711740,0.547667,0,0.475848,0.000000,0.475848,0.475848,0.072924,0.072924,0.000000
5850,2025-11-11 16:00:00+00:00,# [Article ID: 13369] Ditch Big Tech Wallets: ...,1,0.693147,1,0.593831,0.000000,0.593831,0.593831,0.593831,...,0.849824,0.011916,0,0.492953,0.000000,0.492953,0.492953,-0.126375,0.126375,0.000000
5851,2025-11-13 20:00:00+00:00,"# [Article ID: 13370] Unlock Dollars, Lose Con...",19,2.995732,1,0.280821,0.259523,0.664833,-0.332087,0.664833,...,0.849824,0.968565,1,0.298236,0.340481,0.691728,-0.609122,0.004222,0.225657,0.105263


In [38]:
df_news_merged = df_news_merged.rename(columns={"actionable_time": "timestamp"})
pd.concat([
    df_news_merged.head(2),
    df_news_merged.tail(2)
])

,timestamp,merged_content,news_article_count,news_log_article_count,has_news,news_combined_sent_mean,news_combined_sent_std,news_combined_sent_max,news_combined_sent_min,news_combined_sent_max_abs,...,news_max_p_bull_bucket,news_max_p_bear_bucket,news_is_conflict,news_head_sent_mean,news_head_sent_std,news_head_sent_max,news_head_sent_min,news_head_global_diff_mean,news_head_global_abs_diff_mean,news_head_exag_ratio
0,2012-02-28 08:00:00+00:00,# [Article ID: 1] The Wasted Electricity Objec...,1,0.693147,1,0.247521,0.000000,0.247521,0.247521,0.247521,...,0.691526,0.806509,0,0.500018,0.000000,0.500018,0.500018,0.510090,0.510090,1.000000
1,2012-02-28 12:00:00+00:00,# [Article ID: 2] Bitcoin Adoption Opportunity...,1,0.693147,1,0.412206,0.000000,0.412206,0.412206,0.412206,...,0.711740,0.547667,0,0.475848,0.000000,0.475848,0.475848,0.072924,0.072924,0.000000
5850,2025-11-11 16:00:00+00:00,# [Article ID: 13369] Ditch Big Tech Wallets: ...,1,0.693147,1,0.593831,0.000000,0.593831,0.593831,0.593831,...,0.849824,0.011916,0,0.492953,0.000000,0.492953,0.492953,-0.126375,0.126375,0.000000
5851,2025-11-13 20:00:00+00:00,"# [Article ID: 13370] Unlock Dollars, Lose Con...",19,2.995732,1,0.280821,0.259523,0.664833,-0.332087,0.664833,...,0.849824,0.968565,1,0.298236,0.340481,0.691728,-0.609122,0.004222,0.225657,0.105263


In [39]:
dataframes['news'] = df_news_merged
pd.concat([
    dataframes['news'].head(2),
    dataframes['news'].tail(2)
])

,timestamp,merged_content,news_article_count,news_log_article_count,has_news,news_combined_sent_mean,news_combined_sent_std,news_combined_sent_max,news_combined_sent_min,news_combined_sent_max_abs,...,news_max_p_bull_bucket,news_max_p_bear_bucket,news_is_conflict,news_head_sent_mean,news_head_sent_std,news_head_sent_max,news_head_sent_min,news_head_global_diff_mean,news_head_global_abs_diff_mean,news_head_exag_ratio
0,2012-02-28 08:00:00+00:00,# [Article ID: 1] The Wasted Electricity Objec...,1,0.693147,1,0.247521,0.000000,0.247521,0.247521,0.247521,...,0.691526,0.806509,0,0.500018,0.000000,0.500018,0.500018,0.510090,0.510090,1.000000
1,2012-02-28 12:00:00+00:00,# [Article ID: 2] Bitcoin Adoption Opportunity...,1,0.693147,1,0.412206,0.000000,0.412206,0.412206,0.412206,...,0.711740,0.547667,0,0.475848,0.000000,0.475848,0.475848,0.072924,0.072924,0.000000
5850,2025-11-11 16:00:00+00:00,# [Article ID: 13369] Ditch Big Tech Wallets: ...,1,0.693147,1,0.593831,0.000000,0.593831,0.593831,0.593831,...,0.849824,0.011916,0,0.492953,0.000000,0.492953,0.492953,-0.126375,0.126375,0.000000
5851,2025-11-13 20:00:00+00:00,"# [Article ID: 13370] Unlock Dollars, Lose Con...",19,2.995732,1,0.280821,0.259523,0.664833,-0.332087,0.664833,...,0.849824,0.968565,1,0.298236,0.340481,0.691728,-0.609122,0.004222,0.225657,0.105263


#### Sentiment index

In [40]:
dataframes['sentiment_index'].head(5)

,timestamp,value,classification
0,2018-02-01 00:00:00+00:00,30,Fear
1,2018-02-02 00:00:00+00:00,15,Extreme Fear
2,2018-02-03 00:00:00+00:00,40,Fear
3,2018-02-04 00:00:00+00:00,24,Extreme Fear
4,2018-02-05 00:00:00+00:00,11,Extreme Fear


In [41]:
dataframes['sentiment_index'] = convert_timestamp(dataframes['sentiment_index'], 'timestamp')
dataframes['sentiment_index'].head(5)

,timestamp,value,classification
0,2018-02-01 00:00:00+00:00,30,Fear
1,2018-02-02 00:00:00+00:00,15,Extreme Fear
2,2018-02-03 00:00:00+00:00,40,Fear
3,2018-02-04 00:00:00+00:00,24,Extreme Fear
4,2018-02-05 00:00:00+00:00,11,Extreme Fear


In [42]:
dataframes['sentiment_index'].rename(columns=lambda x: f"sentiment_index_{x}" if x != "timestamp" else x, inplace=True)
dataframes['sentiment_index'].head(5)

,timestamp,sentiment_index_value,sentiment_index_classification
0,2018-02-01 00:00:00+00:00,30,Fear
1,2018-02-02 00:00:00+00:00,15,Extreme Fear
2,2018-02-03 00:00:00+00:00,40,Fear
3,2018-02-04 00:00:00+00:00,24,Extreme Fear
4,2018-02-05 00:00:00+00:00,11,Extreme Fear


### Merge

In [43]:
# Convert all timestamps to datetime64[ns] without timezone
for name, df in dataframes.items():
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
        df["timestamp"] = df["timestamp"].dt.tz_convert(None)
        dataframes[name] = df

print("All timestamps coerced to datetime64[ns].")

All timestamps coerced to datetime64[ns].


In [44]:
VALID_HOURS = {0, 4, 8, 12, 16, 20}

def check_h4_timestamp(df, name):
    ts = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

    bad_format = ts.isna().sum()
    bad_hour = (~ts.dt.hour.isin(VALID_HOURS)).sum()

    return bad_format == 0 and bad_hour == 0

# Check all dataframes
ok_flags = {}
for name, df in dataframes.items():
    if "timestamp" in df.columns:
        ok_flags[name] = check_h4_timestamp(df, name)
    else:
        print(f"\n{name}  NO timestamp column")
        ok_flags[name] = False

print("\nH4-ready status:")
error = 0
for name, status in ok_flags.items():
    status_str = "✅" if status else "❌"
    print(f"  {name}: {status_str}")
    if not status:
        error += 1

if error == 0:
    print("All dataframes are H4-ready.")
else:
    print(f"{error} dataframes are NOT H4-ready.")


H4-ready status:
  btc_ohlcv: ✅
  eth_ohlcv: ✅
  market_cap: ✅
  btc_network_activity: ✅
  eth_network_activity: ✅
  mining: ✅
  btc_onchain_metrics: ✅
  news: ✅
  sentiment_index: ✅
All dataframes are H4-ready.


In [45]:
dfs_to_merge = []
for name, df in dataframes.items():
    if "timestamp" in df.columns and ok_flags.get(name, False):
        dfs_to_merge.append(df)

print(f"Number of dataframes used for merge: {len(dfs_to_merge)}")

final_df = reduce(
    lambda left, right: pd.merge(left, right, on="timestamp", how="outer"),
    dfs_to_merge
)

final_df = final_df.sort_values("timestamp").reset_index(drop=True)
dataframes["final_df"] = final_df

print("Final merged shape:", final_df.shape)
final_df.head(3)

Number of dataframes used for merge: 9
Final merged shape: (26209, 61)


,timestamp,BTC_open,BTC_high,BTC_low,BTC_close,BTC_volume,BTC_quote_volume,BTC_trades,ETH_open,ETH_high,...,news_is_conflict,news_head_sent_mean,news_head_sent_std,news_head_sent_max,news_head_sent_min,news_head_global_diff_mean,news_head_global_abs_diff_mean,news_head_exag_ratio,sentiment_index_value,sentiment_index_classification
0,2009-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
# Get global start and end timestamps
start = final_df["timestamp"].min()
end   = final_df["timestamp"].max()
print("Global time range:", start, "->", end)

# Create full H4 index (including timestamps that don't appear in any df)
full_idx = pd.date_range(start=start, end=end, freq="4h")

# Reindex to ensure ALL H4 timestamps are present
final_df = (
    final_df
    .set_index("timestamp")
    .reindex(full_idx)
    .rename_axis("timestamp")
    .reset_index()
)

print("After reindex to full H4 grid:", final_df.shape)
dataframes["final_df"] = final_df
final_df.head(10)

Global time range: 2009-01-03 00:00:00 -> 2025-11-14 00:00:00
After reindex to full H4 grid: (36955, 61)


,timestamp,BTC_open,BTC_high,BTC_low,BTC_close,BTC_volume,BTC_quote_volume,BTC_trades,ETH_open,ETH_high,...,news_is_conflict,news_head_sent_mean,news_head_sent_std,news_head_sent_max,news_head_sent_min,news_head_global_diff_mean,news_head_global_abs_diff_mean,news_head_exag_ratio,sentiment_index_value,sentiment_index_classification
0,2009-01-03 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009-01-03 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009-01-03 08:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2009-01-03 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009-01-03 16:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2009-01-03 20:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2009-01-04 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2009-01-04 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2009-01-04 08:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2009-01-04 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
if "merged_content" in final_df.columns:
    final_df = final_df.drop(columns=["merged_content"])
if "original_ids" in final_df.columns:
    final_df = final_df.drop(columns=["original_ids"])

In [48]:
master_dataset_path = get_master_dataset_file()
final_df.to_csv(master_dataset_path, index=False)
print(f"Saved master dataset to {master_dataset_path}")

Saved master dataset to data/raw/master_dataset_h4_v1.csv
